# Agri-Intelligence: Fine-Tuning Mistral-7B with QLoRA

In [ ]:
!pip install transformers peft trl accelerate bitsandbytes datasets

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer


### Load Base Model and Configuration & Hyperparameters

In [ ]:

# 🧠 Model to fine-tune
model_name = "mistralai/Mistral-7B-Instruct-v0.3"


dataset_name = "Mahesh2841/Agriculture"

# 🆕 Name for the fine-tuned model
new_model = "mistral-7b-agri-qlora"

#---Lora settings
# LoRA rank
lora_r = 8
# Scaling factor for LoRA layers
lora_alpha = 16
# Dropout in LoRA adapters
lora_dropout = 0.1
#------quantization

# Enable 4-bit loading
use_4bit = True
# Data type used for computations
bnb_4bit_compute_dtype = "float16"
# Quantization type ("nf4" recommended)
bnb_4bit_quant_type = "nf4"
# Use nested quantization (double quant)
use_nested_quant = True

#training arguments
output_dir = "./mistral_agri_results"      # checkpoints & logs
num_train_epochs = 3                       # adjust if needed
fp16 = True                                # Colab GPUs support fp16
bf16 = False                               # bf16 only for A100
per_device_train_batch_size = 1            # small to fit T4/Colab
per_device_eval_batch_size = 1
gradient_accumulation_steps = 4            # effective batch = 8
gradient_checkpointing = True
max_grad_norm = 0.3
learning_rate = 2e-4
weight_decay = 0.001
optim = "paged_adamw_32bit"
lr_scheduler_type = "cosine"
max_steps = -1                             # -1 → use num_train_epochs
warmup_ratio = 0.03
group_by_length = True                     # packs similar-length samples
save_strategy = "epoch"
logging_steps = 50


# 5) SFT / Trainer parameters

max_seq_length = 512          # good default for 7B instruct
packing = False                # True only if data has many shorts
device_map = "auto"            # let HF decide best placement
offload_folder = "./offload"

In [ ]:
!pip install -q huggingface_hub


In [ ]:
from huggingface_hub import login

login(token="TOKEN_")

### Dataset Loading & Preprocessing

In [ ]:
# Load dataset
dataset = load_dataset(dataset_name, split="train")

# Shuffle and select first 1000 rows
dataset = dataset.shuffle(seed=42).select(range(1000))
print(dataset)

# 2) Prepare quantization / BitsAndBytes config
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
)

# Optional: warn if bf16 is available
if compute_dtype == torch.float16 and use_4bit:
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        print("=" * 80)
        print("✅ Your GPU supports bfloat16 – you could set bf16=True for faster training")
        print("=" * 80)


# 3) Load base model in 4-bit

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map=device_map,
)
model.config.use_cache = False
model.config.pretraining_tp = 1


# 4) Load tokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


# 5) Prepare LoRA (PEFT) config

from peft import LoraConfig
peft_config = LoraConfig(
    r=lora_r,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    bias="none",
    task_type="CAUSAL_LM",
)


# 6) TrainingArguments

from transformers import TrainingArguments

training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_strategy = "steps",        # save every X steps
    save_steps = 200,               # saves every 100 training steps
    save_total_limit = 3,           # keeps latest 3 checkpoint
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    fp16=fp16,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    group_by_length=group_by_length,
    lr_scheduler_type=lr_scheduler_type,
    report_to="tensorboard",
)


# 7) Define the trainer

# Suppose your dataset is tokenized like this
def tokenize_function(examples):
    # Combine instruction + input
    full_texts = [f"{ins}\n{inp}" if inp else ins for ins, inp in zip(examples["instruction"], examples["input"])]

    # Add the response
    full_texts = [f"{p}\n{resp}" for p, resp in zip(full_texts, examples["response"])]

    # Tokenize
    return tokenizer(
        full_texts,
        truncation=True,
        padding="max_length",
        max_length=max_seq_length
    )

# Apply tokenization
tokenized_dataset = dataset.map(tokenize_function, batched=True)

trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset,
    peft_config=peft_config,
    args=training_arguments,
)



### Model Fine-Tuning (SFT)

In [ ]:
# 8) Launch training
trainer.train()

In [ ]:
!pip install -U bitsandbytes

### 4-bit Quantization & Base Model Loading

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.3",
    device_map="auto",
    load_in_4bit=True,
)

tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.3", trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# Pick a checkpoint folder you want to use (e.g., last one)
checkpoint_path = "./mistral_agri_results/checkpoint-600"  # change as needed

# Load PEFT weights from that checkpoint
model = PeftModel.from_pretrained(base_model, checkpoint_path)
model.eval()



### Merging LoRA Weights & Saving Standalone Model

In [ ]:
# Merge LoRA weights into base model to make standalone
standalone_model = model.merge_and_unload()

# Save merged model
save_path = "./mistral_agri_merged"
standalone_model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"✅ Standalone model saved at {save_path}")


In [ ]:
!pip install -U bitsandbytes

In [ ]:
from huggingface_hub import login
login()  # Enter your HF token when prompted


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import HfApi
import os

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import HfApi
import os

# Paths
model_folder = "./mistral_agri_merged"
username = "chanystrange"
repo_name = "mistral-agri-merged_143"
repo_id = f"{username}/{repo_name}"

# --- Load tokenizer (model itself doesn't need full GPU memory to push) ---
tokenizer = AutoTokenizer.from_pretrained(model_folder)
tokenizer.push_to_hub(repo_id)

# --- Push model to Hub without loading entire model into GPU ---
# This avoids OutOfMemoryError
model = AutoModelForCausalLM.from_pretrained(
    model_folder,
    device_map="auto",  # safe even on T4
    load_in_4bit=True   # memory-efficient
)
model.push_to_hub(repo_id)

print(f"✅ Model and tokenizer successfully pushed to Hugging Face Hub!")
print(f"Link: https://huggingface.co/{repo_id}")


In [ ]:
import torch
torch.cuda.empty_cache()   # Clears cached memory
torch.cuda.reset_peak_memory_stats()


### Testing & Inference

In [ ]:
# 🔹 Imports
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
import torch

# --- Hugging Face repo ID of your pushed model ---
repo_id = "chanystrange/mistral-agri-merged_143"

# --- Clear GPU memory before loading ---
torch.cuda.empty_cache()

# --- 4-bit Quantization Config (memory-efficient) ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",      # recommended for LLMs
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# --- Load tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(repo_id)
tokenizer.pad_token = tokenizer.eos_token

# --- Load model in 4-bit with device_map="auto" ---
model = AutoModelForCausalLM.from_pretrained(
    repo_id,
    device_map="auto",
    quantization_config=bnb_config
)

# --- Create text-generation pipeline ---
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.7
)




In [ ]:
# --- Test prompt ---
prompt = "Explain the advantages of crop rotation."
output = generator(prompt)[0]['generated_text']

print("✅ Generated text:\n", output)

In [ ]:
from transformers import pipeline

# Assuming 'model' and 'tokenizer' are already loaded
qa_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200,   # adjust response length
    do_sample=True,
    temperature=0.7
)

# --- Ask a question ---
question = "What are the advantages of crop rotation in farming?"
response = qa_pipeline(question)[0]['generated_text']

print("✅ Response:\n", response)
